# 📓 Diário de Bordo: Radar Mercado Dados SP
## Sprint 1: Épico [INFRA] - Configuração de Ambiente

**Objetivo:** Provisionar e validar a infraestrutura base do projeto utilizando Docker, garantindo um ambiente isolado para o banco de dados e para a orquestração.

### 1. Arquitetura Escolhida (Task-2)
Para otimizar os recursos locais da máquina, optamos por uma arquitetura enxuta com dois serviços principais:
* [cite_start]1.Um container isolado para o PostgreSQL, que será a nossa Landing Zone (onde os dados do projeto vão ficar). 
* [cite_start]2.Um container do Airflow rodando em modo standalone, que inicializa todos os serviços necessários dele internamente sem consumir toda a sua máquina. 

### 2. Análise do `docker-compose.yml`
Abaixo estão as minhas anotações detalhadas sobre a função de cada linha crítica na criação da nossa infraestrutura:

**Sobre o PostgreSQL (Landing Zone):**
* `image: postgres:15`: A instrução principal. [cite_start]Diz ao Docker para ir até a internet (Docker Hub) e baixar a imagem oficial do banco de dados PostgreSQL, especificamente a versão 15. 
* [cite_start]`volumes:` -> `- postgres_data:/var/lib/postgresql/data`:Ela mapeia uma pasta virtual (postgres_data) para a pasta interna do container onde o Postgres salva as tabelas.Se você desligar o container, os dados ficam a salvo no seu PC. 

**Sobre o Apache Airflow (Orquestrador):**
* [cite_start]`image: apache/airflow:2.8.1`: Baixa a imagem oficial do Apache Airflow (versão 2.8.1, bem recente e estável). 
* [cite_start]`- AIRFLOW__CORE__LOAD_EXAMPLES=False`: O Airflow vem com dezenas de DAGs de "tutorial" que poluem a tela.Essa linha diz para ele não carregar esse lixo, deixando a interface limpa apenas para o seu projeto. 
* [cite_start]`- ./dags:/opt/airflow/dags`: Pega a pasta /dags que você criou hoje e a "espelha" para dentro do container.Qualquer código Python que você salvar no seu VS Code aparecerá automaticamente no Airflow. 

* [cite_start]`- ./scripts:/opt/airflow/scripts`: Faz o mesmo espelhamento para a pasta de scripts. 

* [cite_start]`command: standalone`: O pulo do gato.Em vez de subir 6 containers pesados que o Airflow exige por padrão, esse comando roda todos os serviços internos do Airflow (webserver, scheduler, metadata) em um único processo, economizando muita memória RAM do seu PC. 
* `depends_on: - postgres_landing`: Uma regra de segurança.Diz ao Docker: "Só ligue o Airflow depois que o container do banco de dados já estiver 100% ligado". 

**Sobre Redes (Networking):**
* `driver: bridge`: O tipo de rede. [cite_start]O "bridge" (ponte) permite que containers no mesmo computador conversem entre si pelo nome (ou seja, o Airflow vai conseguir chamar o Postgres pelo nome, sem precisar saber o IP dele). [cite: 13]

### 3. Integração e Acessos (Task-3)
* [cite_start]**Configuração da Connection no Airflow:** Para conectar o orquestrador ao banco de dados pela interface web, a configuração do `Host` exige atenção especial: `Host: postgres_landing_zone` (Aqui está o pulo do gato! Como eles estão na mesma rede Docker, você não usa 'localhost', você usa o nome exato do container que definimos no YAML). 

### 4. Validação de Persistência e Resiliência (Task-4)
O conceito de volumes garante que, mesmo que a infraestrutura seja destruída, os dados permaneçam intactos. O fluxo abaixo comprova essa persistência.

In [ ]:
# 1. criar uma tabela e fazer um insert
docker exec -it postgres_landing_zone psql -U admin -d radar_sp -c "CREATE TABLE teste_volume (id SERIAL, mensagem VARCHAR(100)); INSERT INTO teste_volume (mensagem) VALUES ('Se eu sobreviver, o volume funciona!');" [cite: 15]

# 2. desligar o Docker
docker-compose down 

# 3. religar o docker
docker-compose up 

# 4. lançar comando de consulta na tabela criada anteriormente
docker exec -it postgres_landing_zone psql -U admin -d radar_sp -c "SELECT * FROM teste_volume;" [cite: 16]